# Week 2b.2 — Agent Basics in LangChain

Now that we've covered the API basics in LangChain, we're ready to implement our first agent.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
assert os.getenv("TAVILY_API_KEY"), "No TAVILY_API_KEY found."
print("API keys loaded")

API keys loaded


## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


## 1. The problem

LLMs can serve as the brain but their knowledge and reach are limited.

In [4]:
from pprint import pprint

response = model.invoke("How is Charlie Kirk doing?")
pprint(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'extras': {'signature': 'El4KXAFpFH0TLPocijh495ZoZk/uwzbk92z0nVla+sY1P+1PD+UKO1mkhPmKva14Pmh3fbNt9J2HPUGl9IpXov2cTfgKjMR2gOLK8LS8OBfvkzplSZ9R87c/7VU1saeh'},
  'text': "As an AI, I don't have access to real-time information or events "
          'happening right now, so I cannot provide an update on Charlie '
          "Kirk's current status. For the most up-to-date news, I recommend "
          'checking reliable current news outlets or his official social media '
          'channels.',
  'type': 'text'}]


The model has no clock. Its weights were frozen months ago and nothing in the context tells it the time. It can only decline or guess.

We can surpass this limitations by augmenting the LLM with a tool.

## 2. Defining a tool

A tool starts life as an ordinary Python function.

In [5]:
from datetime import datetime

def get_current_time():
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

In [6]:
get_current_time()

'Tuesday, September 22, 2026 at 04:05 PM'

### Requirements for Tool Definition
To make this usable by a model, LangChain needs three things: 
- a name (start with a verb and describe the core of the task)
- a description in the docstring
- an argument schema. 

The `@tool` decorator builds all three from your function. 

**The docstring is not a comment here; the model reads it to decide when to call the tool. Write it carefully.**

In [7]:
# import the tool decorator from langchain's tools module
from langchain.tools import tool

#TODO: define `get_current_time` as a tool
@tool
def get_current_time() -> str:
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

Inspect what the decorator built:

In [8]:
print(get_current_time.name)
print(get_current_time.description)
print(get_current_time.args)

get_current_time
Return the current local date and time.
{}


In [9]:
get_current_time.invoke({})

'Tuesday, September 22, 2026 at 04:05 PM'

## 3. Give the tool to the model

`bind_tools` attaches tool schemas to the model as a **list** of tools.

The model does not gain the ability to run anything. It only gains the ability to ask.

In [10]:
#TODO: bind the tool to the model
model_with_tools = model.bind_tools([get_current_time])

In [11]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What time is it right now?")
response = model_with_tools.invoke([question])

In [12]:
pprint(response)

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_48324': 'El4KXAFpFH0TexrOyl72ytNSil2z5FzCuJ0X428g+N5+7B0FFw18Uw5lVr5dT3pwH8TLgze5JMCwhjEwb1M+bprqRWqQJdF+Mi2Z0CBLL8hL9OzHKZw+fMMwOEXnUo8M'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0caba-18ac-7121-95a6-b8c61436cc17-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_48324', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 12, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}})


In [13]:
response.pretty_print()

================================== Ai Message ==================================

[]
Tool Calls:
  get_current_time (call_48324)
 Call ID: call_48324
  Args:


Note: `content` is empty. Instead of answering, the model produced a `tool_calls` entry naming our tool.

Remember the model proposes; it never executes.

In [14]:
pprint(response.response_metadata)

{'finish_reason': 'STOP',
 'model_name': 'gemini-3.5-flash-lite',
 'model_provider': 'google_genai',
 'safety_ratings': []}


In [15]:
print(response.tool_calls)

[{'name': 'get_current_time', 'args': {}, 'id': 'call_48324', 'type': 'tool_call'}]



Executing is our job. Run the tool with the call the model requested:

In [16]:
call = response.tool_calls[0]
tool_result = get_current_time.invoke(call)
print(tool_result)

content='Tuesday, September 22, 2026 at 04:06 PM' name='get_current_time' tool_call_id='call_48324'


That produced a `ToolMessage`. Now we can combine the full conversation, including the model's tool call and the tool's result, and send it back:

In [17]:
messages = [question, response, tool_result]

final = model_with_tools.invoke(messages)


In [18]:
pprint(final)

AIMessage(content=[{'type': 'text', 'text': 'It is currently 4:06 PM on Tuesday, September 22, 2026.', 'extras': {'signature': 'El4KXAFpFH0T5GFKTGTm/UFimB2VFZ5jTgxpNy324zujrh8RNjHR94Zy0m7talvq/Y67ZM1msVLpACfmg3h2fNI8C9jbSoai/rETd2TkCN8d3R4S/GeUglPY0R7lyN8p'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0caba-7a51-7110-9405-f7854a8a9072-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 23, 'total_tokens': 101, 'input_token_details': {'cache_read': 0}})


In [19]:
print(final.content)

[{'type': 'text', 'text': 'It is currently 4:06 PM on Tuesday, September 22, 2026.', 'extras': {'signature': 'El4KXAFpFH0T5GFKTGTm/UFimB2VFZ5jTgxpNy324zujrh8RNjHR94Zy0m7talvq/Y67ZM1msVLpACfmg3h2fNI8C9jbSoai/rETd2TkCN8d3R4S/GeUglPY0R7lyN8p'}}]


### Interim Summary

1. The model saw the question and proposed a tool call.
2. Our code executed the tool.
3. The result went back into the conversation.
4. The model read the result and answered.

Model proposes, runtime executes, result returns. That loop is an agent.

## 4. `create_agent`: Defining the Agentic Loop

The loop so far looked like this:
 propose, execute, return, repeat until the model answers. 
 
 Because this is such a common pattern, LangChain provides a dedicated method to run the loop: `create_agent`
 
 - https://reference.langchain.com/python/langchain/agents/factory/create_agent

 Basic signature: `create_model(model, tools, system_prompt)`


In [26]:
system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""

In [27]:
from langchain.agents import create_agent

#TODO: define an agent with model, tool list, and system prompt
agent = create_agent(model=model, 
                    tools=[get_current_time],
                    system_prompt=system_prompt
                    )

In [28]:
result = agent.invoke({"messages": question})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What time is it right now?
================================== Ai Message ==================================

[]
Tool Calls:
  get_current_time (call_152131)
 Call ID: call_152131
  Args:
================================= Tool Message =================================
Name: get_current_time

Tuesday, September 22, 2026 at 04:09 PM
================================== Ai Message ==================================

[{'type': 'text', 'text': 'It is currently 4:09 PM on Tuesday, September 22, 2026.', 'extras': {'signature': 'El4KXAFpFH0TKZ6un6HYNWCafD04XkO25dNXrsjojCvo4t0Hm1aXifiVUND6FUzR01/SRPagaaBM7Pvpr4usrjllVmtWjsCIdAJp0IB5Qjq9dgwJQ1VMhrGzjmLSYWVb'}}]


## 6. Multiple tools

Real agents have several tools and choose between them. Models are bad at date arithmetic, so we'll give our agent a second tool for that.

In [29]:
@tool
def count_days_until_given_date(date: str) -> str:
    """Return the number of days from today until a future date given as YYYY-MM-DD."""
    target = datetime.strptime(date, "%Y-%m-%d").date()
    delta = (target - datetime.now().date()).days
    return f"{delta} days"


In [30]:
agent = create_agent(model=model, 
                    tools=[get_current_time,count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )


In [31]:
question_day = HumanMessage(content="How many days until the midterm on 2026-10-30?")
response = agent.invoke({"messages":question_day })


In [32]:
pprint(response)

{'messages': [HumanMessage(content='How many days until the midterm on 2026-10-30?', additional_kwargs={}, response_metadata={}, id='2726f73a-b4be-49cd-a6fe-c963e88a3226'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'count_days_until_given_date', 'arguments': '{"date": "2026-10-30"}'}, '__gemini_function_call_thought_signatures__': {'call_36458': 'El4KXAFpFH0TPq8MafECms2WjEsHK7bwhwJsBYGRUHklfjeYC+KIM06yeQL11KREwa+kkXk7j8OEE68u8WRg1ju4HZqsAwCW33xsonxoPb3nE6P83XUBrnwT9BYnFmPB'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0cabd-337d-73e0-b4ce-21c26f282696-0', tool_calls=[{'name': 'count_days_until_given_date', 'args': {'date': '2026-10-30'}, 'id': 'call_36458', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 31, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}}),
             

The model picked the right tool and filled the argument from the question. Since executing calls is always the same steps, write a small helper:

## 6. Tools that call real APIs

**Motivation for web search:**
GPT-4.1-mini has a knowledge cutoff date of June 1, 2024.
 - https://developers.openai.com/api/docs/models/gpt-4.1-mini

 Qwen3.5's knowledge cutoff date is late 2024 to early 2025.

 Gemini 3.1 Flash-Lite has a knowledge cutoff date of January 2025.

They won't be able to answer questions about events that may have happened after the cutoff date

In [34]:
# same agent as before
agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )

In [35]:
questions = [HumanMessage("Which country won the FIFA World Cup 2026"), 
            HumanMessage("Who won the Super Bowl 2026?"), 
            HumanMessage("Who is the current president of the US")]

In [36]:
result = agent.invoke({"messages": questions[-1]})
pprint(result)


{'messages': [HumanMessage(content='Who is the current president of the US', additional_kwargs={}, response_metadata={}, id='627732a1-71b1-4011-bf5b-65b3e958474f'),
              AIMessage(content=[{'type': 'text', 'text': 'The current president of the United States is Joe Biden.', 'extras': {'signature': 'El4KXAFpFH0T+/5OtInxsKYxjl+C5yj56eV2eWDllhSYgrEhoERaelVi/8Adz24sWUkT3wH+nhYK9ROrVKVBmfNekjdawKIuO5SVNQue/cyoToASF/ac4aiFyAQYADOJ'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0cabd-e595-7021-a2d6-8a19bb355669-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 11, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}})]}


In [37]:
pprint(result["messages"][-1].content)

[{'extras': {'signature': 'El4KXAFpFH0T+/5OtInxsKYxjl+C5yj56eV2eWDllhSYgrEhoERaelVi/8Adz24sWUkT3wH+nhYK9ROrVKVBmfNekjdawKIuO5SVNQue/cyoToASF/ac4aiFyAQYADOJ'},
  'text': 'The current president of the United States is Joe Biden.',
  'type': 'text'}]


In [38]:
# loop through questions and then invoke the agent
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    print("")

'Which country won the FIFA World Cup 2026'
[{'extras': {'signature': 'El4KXAFpFH0T60MitFGwjGROyxu6UBKZzw6Y61MnvwacHHrMXGdX8dQfyjxhmYSSDnU0VDu/y7Pp21V3MoWllneJ+Cln63pmgYbm5Jn9uSLvZp9JPo695uHNMvL5xmE/'},
  'text': 'The 2026 FIFA World Cup has not taken place yet; it is scheduled to '
          'be held in June and July 2026.',
  'type': 'text'}]

'Who won the Super Bowl 2026?'
[{'extras': {'signature': 'El4KXAFpFH0ThvABMaJ7X//9kPq3dSnqTr7LnjtLahzbtwCF1RE50kp+GWYzzv8RABuU3gs5K5G8O9jZz0ORV/o5IClNFe+TxnTAHwUPfPHLacOfg6anY7lfW4CK+I8N'},
  'text': 'The Super Bowl for 2026 (Super Bowl LX) was won by the **Seattle '
          'Seahawks**, who defeated the New York Jets 29–13 on February 8, '
          '2026.',
  'type': 'text'}]

'Who is the current president of the US'
[{'extras': {'signature': 'El4KXAFpFH0Tadnj2MLpsWLfuL8/mL734Qgy1z84YHbSbFgfJUxOtQSjVdcPhyHw7+N/YN3Dbs+tMLOBGAjJjaamG7R+wTrUICvUCnWQIy8seDGmzSg9BIokNOI5astY'},
  'text': 'The current President of the United States is Joe Biden.'

### Tavily Web Search API

Let's add web search capability to our agent. We'll use Tavily as our web search API.
- https://app.tavily.com/

We'll need to provide an API key in the `.env` file, which has a `TAVILY_API_KEY` field.

#### TavilySearch
Because Tavily is specialized for agentic web search, LangChain supports it.

The `TavilySearch` method queries the Tavily Search API and gets back json.

https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch


In [39]:
from langchain_tavily import TavilySearch
from typing import Dict, Any

@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results


In [40]:
data = search_the_web.invoke({"query":"Who won the super bowl 2026"})

In [41]:
search_docs = data.get("results", data)


In [42]:
pprint(search_docs)

[{'content': 'After a long, lonely offseason, you may have forgotten where the '
             'NFL drama left off after the last big game. While all eyes are '
             'focused on making it to Super Bowl 61 – which is still months '
             "away – here's a little refresher to get back up to speed.\n"
             '\n'
             '## Who won Super Bowl 2026?\n'
             '\n'
             'The Seattle Seahawks defeated the New England Patriots, 29-23, '
             "in Super Bowl 60, which took place at Levi's Stadium in Santa "
             'Clara, California, on Feb. 8, 2026. [...] Seattle Seahawks '
             'defeated the New England Patriots, 29-23, in Super Bowl 60.\n'
             '\n'
             '## Who won Super Bowl 60 MVP?\n'
             '\n'
             'Seattle Seahawks running back Kenneth Walker III was named the '
             'MVP of Super Bowl 60.\n'
             '\n'
             '## Who won Super Bowl 2025?\n'
             '\n'
             'T

Our agent has three tools now:

In [43]:
from langchain.agents import create_agent

agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date, search_the_web],
                    system_prompt = "You're a helpful assistant who answers users' questions concisely. Use the tools available when necessary."
)

In [44]:
result = agent.invoke({"messages": questions[0]})

In [45]:
pprint(result)

{'messages': [HumanMessage(content='Which country won the FIFA World Cup 2026', additional_kwargs={}, response_metadata={}, id='29a16523-f6ab-4e88-a751-8b2945a83eaf'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_44960': 'El4KXAFpFH0TRZrHMfKK/prMGfwN01hm65n0UQrQUX2PTnXbrCb/txue3w5FeKrMZKClLkyZ8vqxFu13mqhSl3Kw94tPNGXpLhbT4W/AixIaGzHkVZKot+4RXznON+7W'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0cac0-6d1a-7f52-b3ab-ca258ad7bcb9-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_44960', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 164, 'output_tokens': 12, 'total_tokens': 176, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='Tuesday, September 22, 2026 at 04:13 PM', nam

In [46]:
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    

'Which country won the FIFA World Cup 2026'
[{'extras': {'signature': 'El4KXAFpFH0TSgTd2pnKZwv2E1GHCeP7RHPXepHZt7gR8ad9ZeVib3u2ZRCEr4WR5hQvch7y4zAoflhOJOfcckgtzW5PagUD6ksKTPFDD2AqS/e31HFUfTfeyQGSpqB0'},
  'text': 'Spain won the 2026 FIFA World Cup, defeating Argentina 1–0 in extra '
          'time in the final on July 19, 2026.',
  'type': 'text'}]
'Who won the Super Bowl 2026?'
[{'extras': {'signature': 'El4KXAFpFH0TQzzd8rqke+Im1ZSVchnBz08WXdVGdlwF2HBgsZm5CmlZFzwUehNKOkZxoACk3+t+QnNnu46qhr1F9SHay+LUapZA93wVYMC2hNsWRZmjSKyIg0HIZ8+6'},
  'text': 'The **Seattle Seahawks** won Super Bowl LX (2026), defeating the '
          'New England Patriots 29–13.',
  'type': 'text'}]
'Who is the current president of the US'
[{'extras': {'signature': 'El4KXAFpFH0TTpT6LGnCNAKA17YqLXjPaLjMlROfCacUofgsQ9J7OmNiIOp1LiPE3oxziSpu3+o9CQcf/K+SDjPfs2DnpINFblT/Xk4kIFB8blxSYJvKzgP+ZpItE1PB'},
  'text': 'The current president of the United States is Donald Trump.',
  'type': 'text'}]


In [47]:
# the response from the model
pprint(result["messages"][-1].content)

[{'extras': {'signature': 'El4KXAFpFH0TTpT6LGnCNAKA17YqLXjPaLjMlROfCacUofgsQ9J7OmNiIOp1LiPE3oxziSpu3+o9CQcf/K+SDjPfs2DnpINFblT/Xk4kIFB8blxSYJvKzgP+ZpItE1PB'},
  'text': 'The current president of the United States is Donald Trump.',
  'type': 'text'}]


## 7. ICA: Build a weather tool

Your turn. [wttr.in](https://wttr.in) is a weather service that takes a city name directly and needs no key. One request:

```
https://wttr.in/Boston?format=j1
```

returns JSON. 

The current conditions live in `data["current_condition"][0]`, with fields including `temp_F`, `weatherDesc`, and `windspeedMiles`.

Boston or Boston,MA are acceptable. If city name matches multiple locations, it'll likely default to the one closest to your IP address.

### Your task:
Write a tool `get_weather(city)` that returns the current conditions for that city. Steps:

1. Build the URL from the city argument and fetch it with `requests.get(...).json()`.
2. Pull out temperature, conditions, and wind, and return a readable string.
3. Write the docstring so the model knows when to reach for this tool.
4. Add it to the agent and ask a question that needs it.

Skeleton:

In [48]:
import requests
data = requests.get("https://wttr.in/Boston?format=j1").json()

In [51]:
pprint(data["current_condition"][0])

{'FeelsLikeC': '10',
 'FeelsLikeF': '50',
 'cloudcover': '98',
 'humidity': '59',
 'observation_time': '07:12 PM',
 'precipInches': '0.0',
 'precipMM': '0.0',
 'pressure': '1028',
 'pressureInches': '30',
 'temp_C': '15',
 'temp_F': '60',
 'uvIndex': '3',
 'visibility': '10',
 'visibilityMiles': '6',
 'weatherCode': '122',
 'weatherDesc': [{'value': 'Overcast '}],
 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}],
 'winddir16Point': 'NE',
 'winddirDegree': '52',
 'windspeedKmph': '23',
 'windspeedMiles': '14'}


In [53]:
import requests

@tool
def get_weather(city: str) -> str:
    """Returns the current weather conditions of the specified city."""
    data = requests.get(f"https://wttr.in/{city}?format=j1").json()
    return data["current_condition"][0]

In [62]:
# Test your tool directly first:
# print(get_weather.invoke({"city": "Boston"}))

# Then give it to the agent:
agent = create_agent(model=model, tools=[get_current_time, count_days_until_given_date, search_the_web, get_weather])
result = agent.invoke({"messages": [HumanMessage(content="Should I bring a jacket in Boston tonight with the current weather?")]})
pprint(result["messages"][-1].content)

[{'extras': {'signature': 'El4KXAFpFH0TLg5i8yoTfIi3SKWESRqiVcUltHjjj9payc8h5WDHyknD5kVZ7qTtUHV4m8qQEAlku6GUJlecci2+8/z9/Y+vEsApDaDUU8CwvWF02ZeIOx5+5wIX/+Vy'},
  'text': 'Yes, you should definitely bring a jacket! While the actual '
          'temperature is around 60°F (15°C), it feels cooler (around 50°F / '
          '10°C) with overcast skies and winds blowing at about 14 mph.',
  'type': 'text'}]
